# 7. Explore annotated MSI datasets

This notebook shows how to inspect public MSI dataset metadata with `DatasetExplorer`.

The examples use [METASPACE](https://metaspace2020.eu/). The same `DatasetExplorer` interface can also be initialized for [PRIDE Archive](https://www.ebi.ac.uk/pride/archive/), but each source defines its own filters and metadata fields.

The notebook covers:

1. [creating a dataset explorer](#12-create-datasetexplorer),
2. [checking supported filters and their available values](#2-available-filters),
3. [filtering datasets](#3-dataset-filtering),
4. [reviewing results and rejection reasons](#33-refine-the-filters),
5. [exporting the selected filters to JSON](#4-export-filters).

No `.imzML` or `.ibd` files are downloaded.

### External documentation

- [METASPACE dataset browser](https://metaspace2020.eu/)
- [PRIDE Archive](https://www.ebi.ac.uk/pride/archive/)
- [PRIDE search tutorial](https://www.ebi.ac.uk/training/online/courses/pride-quick-tour/searching-pride/)

<!-- TODO: Add a hyperlink to the tutorial explaining how to run an exported dataset configuration. -->

## 1. Initialization

### 1.1. Set the repository root

The notebook uses paths relative to the project repository. The following cell finds the nearest parent directory containing `pyproject.toml` and changes the working directory to it.


In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root


PosixPath('/home/maxi7524/repositories/MSIAutoEncoderWrapper')

### 1.2. Create `DatasetExplorer`

`DatasetExplorer` stores the current filters, search results and manually excluded dataset IDs. It sends source-specific operations to the selected dataset source.

The `source` argument selects the database, currently implemented are:
- `"metaspace"` which uses METASPACE;
- `"pride"` which uses PRIDE.


> Remark: 
>
> For METASPACE, `cache_dir` specifies a directory in which the dataset catalogue is additionally saved as `available-datasets.json`. Later sessions can load this local file instead of requesting the catalogue again.
> - When `cache_dir=None`, the catalogue is kept only in memory.
> - Set `refresh_cache=True` to request the catalogue again and replace the local file. This operation retrieves metadata only.


In [2]:
from IPython.display import display

from msi_autoencoder_wrapper.dataset_management.exploration import DatasetExplorer


explorer = DatasetExplorer(
    source="metaspace",
    # Save the METASPACE dataset catalogue in this directory.
    cache_dir="assets/local/datasets/metaspace",
    # Set to True to request the catalogue again and replace the local file.
    refresh_cache=False,
)

# The same interface can be initialized for PRIDE:
# pride_explorer = DatasetExplorer(source="pride")


2026-07-31 11:47:35,810 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 3 implementation module(s) in package 'msi_autoencoder_wrapper.dataset_management.sources.strategies'.
2026-07-31 11:47:38,364 | INFO     | msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace:235 | Loaded 19530 METASPACE catalogue records from assets/local/datasets/metaspace/available-datasets.json


## 2. Available filters

Before filtering datasets, we need to inspect:

1. which filter keys are supported;
2. which values occur in the database.

### 2.1. Inspect filter definitions

`get_available_filters()` returns the filter schema for the selected source. For each filter, the schema **may** contain its expected type, default value, provider API field, predefined choices and information about whether it is applied locally.

Filters mapped to a METASPACE API field are sent to METASPACE. Filters marked as local are applied after the datasets have been returned.


In [3]:
available_filters = explorer.get_available_filters()
available_filters


{'name': {'type': 'string', 'api_field': 'nameMask'},
 'dataset_ids': {'type': 'string | list[string]', 'api_field': 'idMask'},
 'submitter_id': {'type': 'string', 'api_field': 'submitter'},
 'group_id': {'type': 'string', 'api_field': 'group'},
 'project_id': {'type': 'string', 'api_field': 'project'},
 'molecule': {'type': 'string',
  'api_field': 'hasAnnotationMatching.compoundQuery'},
 'polarity': {'type': 'Positive | Negative',
  'api_field': 'polarity',
  'choices': ['Positive', 'Negative']},
 'organism': {'type': 'string', 'api_field': 'organism'},
 'organism_part': {'type': 'string', 'api_field': 'organismPart'},
 'condition': {'type': 'string', 'api_field': 'condition'},
 'growth_conditions': {'type': 'string', 'api_field': 'growthConditions'},
 'analyzer_type': {'type': 'string', 'api_field': 'analyzerType'},
 'ionisation_source': {'type': 'string', 'api_field': 'ionisationSource'},
 'maldi_matrix': {'type': 'string', 'api_field': 'maldiMatrix'},
 'has_optical_image': {'type'

### 2.2. Inspect available values

`get_available_values(filter_key)` lists values currently present in accessible METASPACE datasets.

The returned table contains:

- `value`: the value used in a filter (yes, this serves as parameter ...);
- `label`: the displayed label;
- `count`: the number of datasets containing that value.

Use this method before entering values such as `organism_part`, `condition` or `ionisation_source`. These fields may use labels different from the label expected by the user.

Only filters with a finite set of values can be inspected this way. Free-text filters such as `name` and numerical filters such as `min_annotation_count` do not provide a value table.


In [4]:
# Replace the key with another enumerable entry from `available_filters`.
organism_values = explorer.get_available_values("organism_part")
display(organism_values.head(30))

# Typical additional inspections:
# display(explorer.get_available_values("organism_part").head(30))
# display(explorer.get_available_values("polarity"))
# display(explorer.get_available_values("analyzer_type").head(30))
# display(explorer.get_available_values("ionisation_source").head(30))
# display(explorer.get_available_values("maldi_matrix").head(30))


,value,label,count
0,Kidney,Kidney,4030
1,Brain,Brain,2094
2,Cell Line,Cell Line,969
3,Liver,Liver,854
4,Whole organism,Whole organism,772
5,Lung,Lung,700
6,leaf,leaf,622
7,Breast,Breast,506
8,Root,Root,457
9,root,root,413


### 2.3. Search within available values

`get_available_values()` returns a pandas `DataFrame`. Use standard pandas filtering to find labels containing a selected phrase.

The following example searches values available for `organism_part`. It does not filter datasets.


In [5]:
organism_part_values = explorer.get_available_values("organism_part")

search_term = "brain"
matching_organism_parts = organism_part_values[
    organism_part_values["label"].str.contains(
        search_term,
        case=False,
        na=False,
        regex=False,
    )
]

display(matching_organism_parts.head(30))


,value,label,count
1,Brain,Brain,2094
60,brain,brain,35
69,Brain (CSF),Brain (CSF),30
121,Brain | Lung | Liver | Heart | Kidney | Muscle,Brain | Lung | Liver | Heart | Kidney | Muscle,13
152,"Liver, kidney, Brain, Spleen","Liver, kidney, Brain, Spleen",8
212,Homogenized Brain,Homogenized Brain,4
251,Brain Hippocampus,Brain Hippocampus,2
305,Brain (occipital),Brain (occipital),1


## 3. Dataset filtering

### 3.1. Define filters

Pass a dictionary of filter names and values to `explorer.filter(...)`.

Start with a small number of filters. After reviewing the returned datasets, add further conditions.

The example below uses:

- `organism` and `organism_part` to select biological material;
- optional acquisition filters;
- optional annotation-count filters;
- optional molecular statistics;
- `exclude_dataset_ids` for dataset IDs that should not be included.

#### Additional molecular statistics (important)

By default, the result contains annotation counts but not molecule-level statistics.

Set `include_molecule_stats=True` to calculate:

- `molecule_count`: the number of distinct formula–adduct pairs in a dataset;
- `unique_molecule_count`: the number of formula–adduct pairs found only in that dataset within the current result set;
- `unique_molecules`: labels of these pairs.

`min_molecule_count` and `min_unique_molecule_count` use these calculated values as filters.

This requires retrieving annotation identities and therefore performs more API work than filtering only by catalogue metadata. (It takes some time to obtain all of them)

> It still does not download ion images, `.imzML` files or `.ibd` files.


In [6]:
broad_filters = {
    # Biological metadata
    "organism": "Mouse",
    "organism_part": "Brain",

    # Acquisition metadata
    # "polarity": "Positive",
    # "ionisation_source": "MALDI",

    # Annotation filters
    # "status": "FINISHED",
    # "annotation_fdr": 0.05,
    # "has_optical_image": True,
    # "min_annotation_count": 100,

    # Additional molecular statistics
    # "include_molecule_stats": True,
    # "include_spatial_annotation_stats": True, # REMARK: it filters unique values based on `annotation_fdr`
    # "min_molecule_count": 100,
    # "min_unique_molecule_count": 1,

    # Dataset IDs excluded from the selection
    "exclude_dataset_ids": [],
}

results = explorer.filter(broad_filters)

display(results)
print(f"Found {len(results)} datasets")


2026-07-31 11:47:46,078 | INFO     | msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace:215 | METASPACE discovery accepted 224 datasets and rejected 0 datasets
2026-07-31 11:47:46,082 | INFO     | msi_autoencoder_wrapper.dataset_management.exploration.dataset_explorer:113 | Explorer retained 224 records from source metaspace


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,spatial_annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-07-22_19h07m09s,Washed Brain - Section 14 (m/z 480 - 1000),metaspace,None,https://metaspace2020.eu/dataset/2026-07-22_19...,Mouse,Brain,N/A,,,...,None,None,None,None,None,None,None,None,,False
1,2026-07-22_19h08m34s,Washed Brain - Section 16 (m/z 480 - 1000),metaspace,None,https://metaspace2020.eu/dataset/2026-07-22_19...,Mouse,Brain,N/A,,,...,None,None,None,None,None,None,None,None,,False
2,2026-07-22_19h08m05s,Washed Brain - Section 16 (m/z 70 - 480),metaspace,None,https://metaspace2020.eu/dataset/2026-07-22_19...,Mouse,Brain,N/A,,,...,None,None,None,None,None,None,None,None,,False
3,2026-07-22_19h06m31s,Washed Brain - Section 14 (m/z 70 - 480),metaspace,None,https://metaspace2020.eu/dataset/2026-07-22_19...,Mouse,Brain,N/A,,,...,None,None,None,None,None,None,None,None,,False
4,2026-07-22_19h01m32s,Unwashed Brain - Section 15 (m/z 70 - 480),metaspace,None,https://metaspace2020.eu/dataset/2026-07-22_19...,Mouse,Brain,N/A,,,...,None,None,None,None,None,None,None,None,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,2021-09-30_11h26m36s,EY_210929_EDI-27jul21_p56-Male-642-13-9-B_50um...,metaspace,None,https://metaspace2020.eu/dataset/2021-09-30_11...,Mouse,Brain,Wildtype,,,...,None,None,None,None,None,None,None,None,,False
220,2021-07-01_13h29m48s,Control - BT474 Brain Met Tube T3 DAN,metaspace,None,https://metaspace2020.eu/dataset/2021-07-01_13...,Mouse,Brain,Cancer Tumor,,,...,None,None,None,None,None,None,None,None,,False
221,2021-07-01_13h28m24s,Control - BT474 Brain Met Tube T1 DAN,metaspace,None,https://metaspace2020.eu/dataset/2021-07-01_13...,Mouse,Brain,Cancer Tumor,,,...,None,None,None,None,None,None,None,None,,False
222,2021-03-29_18h30m44s,la imaging,metaspace,None,https://metaspace2020.eu/dataset/2021-03-29_18...,mouse,brain,cancer,,,...,None,None,None,None,None,None,None,None,,False


Found 224 datasets


### 3.2. Review and summarize results

Each row of the returned table represents one accepted dataset.

> The result table contains the following groups of columns (for METASPACE):

#### Dataset identification

* `dataset_id` — identifier used by the selected source,
* `name` — dataset name,
* `source` — source adapter used for the query,
* `project_url` — link to the dataset or project page,
* `submitter` — dataset submitter,
* `group` — group associated with the dataset,
* `projects` — METASPACE projects associated with the dataset.

#### Biological metadata

* `organisms` — organism specified in the dataset metadata,
* `organism_parts` — sampled organ or tissue,
* `condition` — biological or experimental condition,
* `growth_conditions` — conditions under which the sample was grown or maintained.

#### Acquisition and image metadata

* `polarity` — positive or negative ion mode;
* `processing_status` — processing status reported by the source,
* `image_size` — spatial dimensions reported for the MSI image,
* `pixel_count` — number of acquired pixels,
* `has_optical_image` — whether the dataset contains an associated optical image.

#### Annotation information

* `databases` — molecular databases used for annotation,
* `annotation_count` — total number of METASPACE annotation results at `annotation_fdr`, summed across annotation databases. 
    > REMARK:
    > 
    > It is not the number of annotated pixels and not the number of unique molecules.

#### Additional molecular statistics

These columns are populated only when `include_molecule_stats=True` or when a molecular-count filter requires their calculation:

* `molecule_count` — number of distinct `(sumFormula, adduct)` pairs detected in the dataset,
* `unique_molecule_count` — number of those pairs occurring only in this dataset among the datasets returned by the current query,
* `unique_molecules` — list of molecule–adduct pairs counted by `unique_molecule_count`.
    > Missing values mean that the source did not provide the corresponding metadata or that the optional statistic was not calculated.

The following cell summarizes categorical columns with `value_counts()` and numerical columns with `describe()`.

#### Spatial annotation statistics

These columns are populated only when `include_spatial_annotation_stats=True`.

Spatial statistics are calculated from METASPACE ion images for annotations satisfying `annotation_fdr`. Intensity magnitude is not interpreted and no intensity threshold is applied. A spatial position is considered annotated when at least one qualifying annotation has a finite, non-zero signal at that position.

* `annotated_pixel_count` — number of distinct acquired spatial positions with signal from at least one annotation satisfying `annotation_fdr`. A pixel is counted only once, even if it contains multiple annotations, adducts, or results from multiple molecular databases,
* `unannotated_pixel_count` — number of acquired pixels without signal from any qualifying METASPACE annotation. It is calculated as `pixel_count - annotated_pixel_count`,
* `annotated_pixel_fraction` — fraction of acquired pixels with at least one qualifying annotation, calculated as `annotated_pixel_count / pixel_count`,
* `spatial_annotation_fdr` — FDR threshold used to select annotations for the spatial calculation,
* `spatial_annotation_count` — number of annotation ion images included in the spatial union,
* `spatial_annotation_database_count` — number of molecular databases that contributed at least one annotation ion image,
* `spatial_stats_status` — status of the spatial calculation. `complete` means that both annotated and unannotated pixel statistics were calculated successfully. `missing_acquired_pixel_count` means that annotated pixels were counted, but the unannotated count and fraction could not be calculated because acquisition geometry was unavailable.

> REMARK:
>
> `unannotated_pixel_count` does not necessarily represent biological background. It includes every acquired pixel without a METASPACE annotation satisfying the selected FDR, including pixels that may contain unidentified molecular signals.
>
> The rectangular ion-image size is not used as the acquired-pixel count because it may contain spatial positions that were not measured. The calculation uses `pixel_count` reported in METASPACE acquisition geometry.
>
> Spatial annotation statistics require downloading ion images for the qualifying annotations and are therefore substantially more expensive than `annotation_count` and `molecule_count`.


In [7]:
# Categorical summaries
categorical_columns = [
    "organism_parts",
    "condition",
    "polarity",
    "processing_status",
    "has_optical_image",
    "databases",
]

for column in categorical_columns:
    if column not in results:
        continue
    print(f"\n{column}")
    display(results[column].value_counts(dropna=False).head(20))

# Quantitative summaries
quantitative_columns = [
    "pixel_count",
    "annotation_count",
    "molecule_count",
    "unique_molecule_count",
]

available_quantitative_columns = [
    column
    for column in quantitative_columns
    if column in results and results[column].notna().any()
]

if available_quantitative_columns:
    display(results[available_quantitative_columns].describe().T)
else:
    print("No quantitative summary fields are populated for this query.")



organism_parts


organism_parts
Brain    206
brain     18
Name: count, dtype: int64


condition


condition
Ischemia           68
Wildtype           53
Diseased           36
N/A                28
unknown            11
Wild type           7
Fresh frozen        5
Knockout            5
Healthy             3
Control             2
Cancer Tumor        2
Control/Disease     1
Control-healthy     1
None                1
cancer              1
Name: count, dtype: int64


polarity


...


has_optical_image


has_optical_image
False    223
True       1
Name: count, dtype: int64


databases


databases
HMDB v4, LipidMaps 2017-12-12, SwissLipids 2018-02-02, KEGG v1              68
HMDB v4                                                                     49
HMDB v4, LipidMaps 2017-12-12                                               31
HMDB v4, HMDB-endogenous v4                                                 13
HMDB v4, LipidMaps 2017-12-12, SwissLipids 2018-02-02                       13
HMDB v4, HMDB-endogenous v4, CoreMetabolome v3                               8
CoreMetabolome v3, HMDB-endogenous v4, SwissLipids 2018-02-02, HMDB v4       6
BraChemDB 2018-01, SwissLipids 2018-02-02, HMDB v4, DrugBank 5.1             4
HMDB v4, LipidMaps 2017-12-12, SwissLipids 2018-02-02, CoreMetabolome v3     4
HMDB v4, KEGG v1                                                             4
NGlycDB v1, HMDB v4, SwissLipids 2018-02-02, KEGG v1                         3
HMDB v4, SwissLipids 2018-02-02, LipidMaps 2017-12-12, CoreMetabolome v3     3
CoreMetabolome v3, HMDB v4, LipidMaps 2017

,count,mean,std,min,25%,50%,75%,max
pixel_count,224.0,24257.888393,33101.608633,168.0,5417.0,22860.0,28445.75,359468.0
annotation_count,224.0,996.625000,1386.707034,0.0,23.0,343.5,1435.75,5998.0


### 3.3. Refine the filters

Use values found in the catalogue and in the previous result table to define a narrower query.

The example below combines biological metadata, acquisition metadata, annotation count and molecular statistics.

`name` is a **free-text** METASPACE filter. It matches dataset names and does not provide a list of available values.


In [8]:
filters = {
    # Biological and acquisition metadata
    "organism": "Mouse",
    "name": "liver",
    "organism_part": "Liver",
    "polarity": "Negative",
    "condition": "Wild type",

    # Annotation filters and molecular statistics
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": True, # REMARK: it filters unique values based on `annotation_fdr`

    # Dataset IDs excluded from the selection
    "exclude_dataset_ids": [],
}

results = explorer.filter(filters)

display(results)
print(f"Accepted {len(results)} datasets")


100%|████████████████████████████████████████| 14/14 [00:00<00:00, 26.87it/s]


2026-07-31 11:48:07,711 | INFO     | msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace:215 | METASPACE discovery accepted 2 datasets and rejected 0 datasets
2026-07-31 11:48:07,714 | INFO     | msi_autoencoder_wrapper.dataset_management.exploration.dataset_explorer:113 | Explorer retained 2 records from source metaspace


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,spatial_annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2024-06-12_15h42m44s,NEDC_imaging_liver_L1_replicate1,metaspace,None,https://metaspace2020.eu/dataset/2024-06-12_15...,mouse,Liver,Wild type,,,...,2391,0.976254,0.1,40,3,complete,24,12,"C10H16N3O6S[M]-, C10H17N3O6S-H, C12H14Cl2N2-H,...",False
1,2023-10-25_16h42m47s,NEDC_imaging_liver,metaspace,None,https://metaspace2020.eu/dataset/2023-10-25_16...,mouse,liver,Wild type,,,...,3099,0.975902,0.1,45,3,complete,28,16,"C12H14N2+Cl, C12H22O11+Cl, C16H32O2+Cl, C16H33...",False


Accepted 2 datasets


### 3.4. Inspect rejected datasets

`explorer.rejected()` returns datasets rejected by local filters. The table may contain the dataset ID, dataset name, rejection reason and METASPACE URL.

It does not contain datasets removed directly by METASPACE API filters, because such datasets **are not returned to the explorer**.

Use this table to check which local threshold removed each dataset.


In [9]:
rejected = explorer.rejected()

display(rejected)
print(f"Rejected locally: {len(rejected)} datasets")


""


Rejected locally: 0 datasets


### 3.5. Inspect full metadata

The result table contains selected fields in a common format. To inspect the full metadata returned for one dataset, call `get_dataset_metadata(dataset_id)` on the selected source.

The `project_url` column links to the dataset page in METASPACE.


In [10]:
if not results.empty:
    dataset_id = results.iloc[0]["dataset_id"]
    dataset_record = explorer.source.get_dataset_metadata(dataset_id)

    display(dataset_record)
    print(results.iloc[0]["project_url"])
else:
    print("The current query returned no datasets.")


{'dataset_id': '2024-06-12_15h42m44s',
 'name': 'NEDC_imaging_liver_L1_replicate1',
 'metadata': {'Data_Type': 'Imaging MS',
  'Sample_Information': {'Condition': 'Wild type',
   'Organism': 'mouse',
   'Organism_Part': 'Liver',
   'Sample_Growth_Conditions': ''},
  'Sample_Preparation': {'MALDI_Matrix': 'n-(1-naphthyl)ethylenediamine dihydrochloride (NEDC)',
   'Tissue_Modification': 'None',
   'Sample_Stabilisation': 'Fresh frozen',
   'MALDI_Matrix_Application': 'SunChrom Sprayer',
   'Solvent': '70/25/5 MeOH/ACN/water'},
  'MS_Analysis': {'Polarity': 'Negative',
   'Ionisation_Source': 'MALDI',
   'Analyzer': 'Orbitrap',
   'Detector_Resolving_Power': {'Resolving_Power': 120000, 'mz': 200},
   'Pixel_Size': {'Xaxis': 30, 'Yaxis': 30}},
  'Additional_Information': {'Supplementary': ''},
  'project_url': 'https://metaspace2020.eu/dataset/2024-06-12_15h42m44s',
  'provider_metadata': {'id': '2024-06-12_15h42m44s',
   'name': 'NEDC_imaging_liver_L1_replicate1',
   'uploadDT': '2024-06-

https://metaspace2020.eu/dataset/2024-06-12_15h42m44s


### 3.6. Exclude reviewed datasets

Use manual exclusions when a dataset passes the filters but should not be used in the experiment.

- `exclude(dataset_ids)` removes IDs from the displayed selection and adds them to the exported exclusion list;
- `include(dataset_ids)` removes IDs from that exclusion list;
- `results(include_excluded=True)` displays accepted and manually excluded datasets together.

An ID can be excluded only after it has appeared in the current search results.


In [11]:
excluded_dataset_ids = []

if excluded_dataset_ids:
    explorer.exclude(excluded_dataset_ids)

display(explorer.results())


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,spatial_annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2024-06-12_15h42m44s,NEDC_imaging_liver_L1_replicate1,metaspace,None,https://metaspace2020.eu/dataset/2024-06-12_15...,mouse,Liver,Wild type,,,...,2391,0.976254,0.1,40,3,complete,24,12,"C10H16N3O6S[M]-, C10H17N3O6S-H, C12H14Cl2N2-H,...",False
1,2023-10-25_16h42m47s,NEDC_imaging_liver,metaspace,None,https://metaspace2020.eu/dataset/2023-10-25_16...,mouse,liver,Wild type,,,...,3099,0.975902,0.1,45,3,complete,28,16,"C12H14N2+Cl, C12H22O11+Cl, C16H32O2+Cl, C16H33...",False


## 4. Export filters

### 4.1. Write the JSON file

`export_config(path)` writes the current filters to JSON. Dataset IDs added with `exclude(...)` are included in `exclude_dataset_ids`.

The file contains only the dataset search configuration.


In [12]:
output_path = Path(
    "assets/configs/datasets/metaspace_mouse_liver.json"
)

exported_path = explorer.export_config(output_path)
print(exported_path)


2026-07-31 11:49:16,574 | INFO     | msi_autoencoder_wrapper.dataset_management.exploration.dataset_explorer:188 | Exported dataset query configuration to assets/configs/datasets/metaspace_mouse_liver.json
assets/configs/datasets/metaspace_mouse_liver.json


### 4.2. Check the exported file

Read the JSON file and inspect its content before using it in another command.


In [13]:
import json

exported_config = json.loads(exported_path.read_text(encoding="utf-8"))
exported_config


{'organism': 'Mouse',
 'name': 'liver',
 'organism_part': 'Liver',
 'polarity': 'Negative',
 'condition': 'Wild type',
 'annotation_fdr': 0.1,
 'min_annotation_count': 1,
 'include_molecule_stats': True,
 'include_spatial_annotation_stats': True,
 'exclude_dataset_ids': []}

## 5. Next step

The output of this notebook is the exported JSON file.

<!-- # TODO: Add a hyperlink to the tutorial explaining how to run an exported dataset configuration. -->

Running the query configuration and downloading `.imzML` and `.ibd` files are described in [the next tutorial](08_metaspace_dataset_download_and_merge.ipynb).
